# Lecture Notebook: Multimodal Building Sensor Analysis and CO₂ Prediction

This notebook uses a **synthetic building-management dataset** with 30-minute sampling over 2 years.

## Goal
We will use multiple sensor streams to understand building behavior and build a **small prediction task**:

> Predict **indoor CO₂ concentration 30 minutes ahead**.

## Sensors used
- `indoor_humidity_pct`
- `indoor_light_lux`
- `indoor_temp_c`
- `outdoor_temp_c`
- `outdoor_precip_mm_30min`
- `co2_ppm`
- `motion_detected`
- `power_consumption_kw`
- `window_open_state`
- `hvac_state`

## Teaching flow
1. Explore the data
2. Build features
3. Train baseline models
4. Compare results
5. Discuss which signals mattered most


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Optional: XGBoost if available
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

DATA_PATH = Path("./data/building_multisensor_2years_30min_v2.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print("Rows:", len(df))
print("Time range:", df["timestamp"].min(), "to", df["timestamp"].max())
df.head()


## 1. Explore the data

We first inspect one week of data to build intuition.

Questions:
- How do sensors change over a day?
- How do occupancy-related signals like motion, light, power, and CO₂ interact?
- Can we visually distinguish building use patterns?


In [ ]:
week = df.set_index("timestamp").loc["2024-03-04":"2024-03-10"].copy()

week[["co2_ppm", "indoor_temp_c", "outdoor_temp_c"]].plot(figsize=(14, 4), title="One week: CO₂ and temperature")
plt.ylabel("Value")
plt.show()

week[["indoor_light_lux", "power_consumption_kw"]].plot(figsize=(14, 4), title="One week: indoor light and power")
plt.ylabel("Value")
plt.show()

week[["motion_detected", "window_open_state"]].plot(figsize=(14, 3), title="One week: motion and window state")
plt.ylabel("State")
plt.show()


### Weekdays vs weekends

A building often behaves very differently on weekdays and weekends.  
We compare average profiles over the full dataset.


In [ ]:
tmp = df.copy()
tmp["hour_float"] = tmp["timestamp"].dt.hour + tmp["timestamp"].dt.minute / 60
tmp["is_weekend"] = tmp["timestamp"].dt.weekday >= 5

features_to_compare = ["co2_ppm", "power_consumption_kw", "indoor_light_lux", "motion_detected"]

fig, axes = plt.subplots(len(features_to_compare), 1, figsize=(12, 12), sharex=True)
for ax, col in zip(axes, features_to_compare):
    weekday_profile = tmp.loc[~tmp["is_weekend"]].groupby("hour_float")[col].mean()
    weekend_profile = tmp.loc[tmp["is_weekend"]].groupby("hour_float")[col].mean()
    ax.plot(weekday_profile.index, weekday_profile.values, label="Weekday")
    ax.plot(weekend_profile.index, weekend_profile.values, label="Weekend")
    ax.set_title(f"Average daily profile: {col}")
    ax.legend()
    ax.grid(alpha=0.3)
plt.xlabel("Hour of day")
plt.tight_layout()
plt.show()


### Correlations

We inspect pairwise correlations for the numeric variables.

This is only a first hint:
- correlation does not imply causation
- time-lagged relationships may be stronger than same-time correlations


In [ ]:
numeric_cols = [
    "indoor_humidity_pct",
    "indoor_light_lux",
    "indoor_temp_c",
    "outdoor_temp_c",
    "outdoor_precip_mm_30min",
    "co2_ppm",
    "motion_detected",
    "power_consumption_kw",
    "window_open_state",
]
corr = df[numeric_cols].corr(numeric_only=True)
corr.round(2)


In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(label="Correlation")
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()


### Inspect one rain event

Rain is interesting because it can influence:
- outdoor light
- window opening behavior
- indoor humidity
- indirectly, indoor comfort and power use


In [ ]:
rain_candidates = df.loc[df["outdoor_precip_mm_30min"] > 0, "timestamp"]
rain_start = rain_candidates.iloc[100]  # deterministic choice somewhere inside the series
start = rain_start - pd.Timedelta(hours=18)
end = rain_start + pd.Timedelta(hours=30)

rain_event = df.set_index("timestamp").loc[start:end].copy()

rain_event[["outdoor_precip_mm_30min"]].plot(figsize=(14, 3), title="Rain event: precipitation")
plt.show()

rain_event[["indoor_humidity_pct", "co2_ppm", "power_consumption_kw"]].plot(
    figsize=(14, 4), title="Rain event: humidity, CO₂, and power"
)
plt.show()

rain_event[["indoor_light_lux", "window_open_state"]].plot(
    figsize=(14, 4), title="Rain event: indoor light and window state"
)
plt.show()

print("Rain event window:", start, "to", end)


## 2. Build features

We will predict:

- **Target:** `co2_ppm` at the **next time step** (30 minutes ahead)

Feature engineering:
- lag features: previous 1, 2, 4 steps
- rolling means over the last 2 hours
- hour of day encoded cyclically
- weekday / weekend indicator
- one-hot encoding of HVAC state

This is still simple enough for teaching, but already realistic.


In [ ]:
feat = df.copy()

# Time-based features
feat["hour"] = feat["timestamp"].dt.hour + feat["timestamp"].dt.minute / 60
feat["dayofweek"] = feat["timestamp"].dt.weekday
feat["is_weekend"] = (feat["dayofweek"] >= 5).astype(int)

# Cyclic encoding for hour
feat["hour_sin"] = np.sin(2 * np.pi * feat["hour"] / 24)
feat["hour_cos"] = np.cos(2 * np.pi * feat["hour"] / 24)

base_features = [
    "indoor_humidity_pct",
    "indoor_light_lux",
    "indoor_temp_c",
    "outdoor_temp_c",
    "outdoor_precip_mm_30min",
    "co2_ppm",
    "motion_detected",
    "power_consumption_kw",
    "window_open_state",
]

# Lags: previous 1, 2, 4 steps (30, 60, 120 minutes)
for col in base_features:
    for lag in [1, 2, 4]:
        feat[f"{col}_lag{lag}"] = feat[col].shift(lag)

# Rolling mean over last 2 hours = 4 samples
for col in base_features:
    feat[f"{col}_roll4"] = feat[col].shift(1).rolling(4).mean()

# Target: predict next-step CO2
feat["target_co2_next"] = feat["co2_ppm"].shift(-1)

# Encode HVAC state
feat = pd.get_dummies(feat, columns=["hvac_state"], drop_first=False)

# Drop rows with NaNs from shifting / rolling
feat = feat.dropna().reset_index(drop=True)

print("Feature rows after engineering:", len(feat))
feat.head()


### Train/test split

Because this is time-series data, we **must not** shuffle randomly.

We use:
- earlier portion for training
- later portion for testing

This better reflects real forecasting.


In [ ]:
# Keep timestamp for plotting later
feature_cols = [c for c in feat.columns if c not in ["timestamp", "target_co2_next"]]

split_idx = int(len(feat) * 0.8)

train_df = feat.iloc[:split_idx].copy()
test_df = feat.iloc[split_idx:].copy()

X_train = train_df[feature_cols]
y_train = train_df["target_co2_next"]

X_test = test_df[feature_cols]
y_test = test_df["target_co2_next"]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("Train end:", train_df["timestamp"].max())
print("Test start:", test_df["timestamp"].min())


## 3. Baseline prediction models

We compare:
1. Naive baseline: next CO₂ = current CO₂
2. Linear regression
3. Random forest
4. XGBoost (if available)

The naive baseline is important because many time-series targets are strongly autocorrelated.


In [ ]:
results = []

# 1. Naive baseline
y_pred_naive = test_df["co2_ppm"].values
mae = mean_absolute_error(y_test, y_pred_naive)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_naive))
results.append({"model": "Naive (next = current)", "MAE": mae, "RMSE": rmse})

# 2. Linear Regression
linreg = LinearRegression()
linreg.fit(X_train, y_train)
y_pred_lin = linreg.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_lin)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lin))
results.append({"model": "Linear Regression", "MAE": mae, "RMSE": rmse})

# 3. Random Forest
rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
results.append({"model": "Random Forest", "MAE": mae, "RMSE": rmse})

# 4. XGBoost if available
if HAS_XGB:
    xgb = XGBRegressor(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42
    )
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred_xgb)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
    results.append({"model": "XGBoost", "MAE": mae, "RMSE": rmse})

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
results_df


## 4. Compare results

We compare models numerically and visually.


In [ ]:
results_df.plot(
    x="model", y=["MAE", "RMSE"], kind="bar", figsize=(10, 4), title="Model comparison"
)
plt.ylabel("Error")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# Visual comparison over a 2-day segment from the test set
plot_start = test_df["timestamp"].iloc[200]
plot_end = plot_start + pd.Timedelta(days=2)

mask = (test_df["timestamp"] >= plot_start) & (test_df["timestamp"] <= plot_end)

plt.figure(figsize=(14, 5))
plt.plot(test_df.loc[mask, "timestamp"], y_test.loc[mask], label="True CO₂")
plt.plot(test_df.loc[mask, "timestamp"], y_pred_naive[mask.values], label="Naive", alpha=0.9)
plt.plot(test_df.loc[mask, "timestamp"], y_pred_lin[mask.values], label="Linear Regression", alpha=0.9)
plt.plot(test_df.loc[mask, "timestamp"], y_pred_rf[mask.values], label="Random Forest", alpha=0.9)
if HAS_XGB:
    plt.plot(test_df.loc[mask, "timestamp"], y_pred_xgb[mask.values], label="XGBoost", alpha=0.9)

plt.title("2-day prediction comparison on the test set")
plt.ylabel("CO₂ (ppm)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Discussion and interpretation

Now we ask:
- which sensors mattered most?
- does motion help?
- do windows improve prediction?
- is precipitation indirectly useful?

We use feature importance from tree-based models and a simple ablation study.


In [ ]:
# Feature importance from Random Forest
rf_importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
rf_importance.head(20)


In [ ]:
rf_importance.head(15).sort_values().plot(kind="barh", figsize=(10, 6), title="Top 15 Random Forest feature importances")
plt.tight_layout()
plt.show()


In [ ]:
def evaluate_feature_set(drop_cols=None):
    drop_cols = drop_cols or []
    cols = [c for c in feature_cols if c not in drop_cols]
    Xtr = train_df[cols]
    Xte = test_df[cols]
    model = RandomForestRegressor(
        n_estimators=120,
        max_depth=12,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    return {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred))
    }

ablation_results = []

# Full model
full = evaluate_feature_set([])
ablation_results.append({"setting": "Full feature set", **full})

# Remove motion
drop_motion = [c for c in feature_cols if "motion_detected" in c]
ablation_results.append({"setting": "Without motion", **evaluate_feature_set(drop_motion)})

# Remove window signals
drop_window = [c for c in feature_cols if "window_open_state" in c]
ablation_results.append({"setting": "Without windows", **evaluate_feature_set(drop_window)})

# Remove precipitation signals
drop_rain = [c for c in feature_cols if "outdoor_precip_mm_30min" in c]
ablation_results.append({"setting": "Without precipitation", **evaluate_feature_set(drop_rain)})

# Remove power signals
drop_power = [c for c in feature_cols if "power_consumption_kw" in c]
ablation_results.append({"setting": "Without power", **evaluate_feature_set(drop_power)})

# Remove current CO2 and its history
drop_co2 = [c for c in feature_cols if "co2_ppm" in c]
ablation_results.append({"setting": "Without CO₂ history", **evaluate_feature_set(drop_co2)})

ablation_df = pd.DataFrame(ablation_results).sort_values("RMSE").reset_index(drop=True)
ablation_df


In [ ]:
ablation_df.plot(x="setting", y=["MAE", "RMSE"], kind="bar", figsize=(12, 4), title="Ablation study")
plt.ylabel("Error")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()
